# 🧪 Lab 3 — Frequency Domain, Image Gradients & Image Pyramids

| | |
|---|---|
| **Sessions** | 9 – 13 (Frequency domain analysis · Image gradients · Multi-scale analysis & pyramids) |
| **Weeks** | 4 – 5 |
| **Learning Outcome** | **LO 2.2** Frequency domain analysis and image gradients |
| **Type** | Practice assignment — do it **before** your Lab workshop (Session 13) |
| **Estimated time** | 90 – 120 minutes |

---

### 🎯 Objectives
1. Compute and read the **2-D DFT magnitude spectrum**.
2. Filter in the **frequency domain** (ideal low/high pass), see the **convolution theorem**, and explain **ringing**.
3. Compute **image gradients** (Sobel) and detect edges; explain why we smooth before differentiating (**DoG**).
4. Demonstrate **aliasing** when sampling below the Nyquist rate.
5. Build a **Gaussian / Laplacian pyramid** and verify **perfect reconstruction**.

> 💡 Fill in every `# ==== YOUR CODE HERE ====` block and answer the markdown
> questions. **Run All** before submitting.



## Setup



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)
print("✅ numpy", np.__version__, "| opencv", cv2.__version__)



## Part 1 · The 2-D Fourier transform

**Recap.** An image is a sum of 2-D sinusoids at different frequencies and
orientations. The **DFT** $F(u,v)=\sum_{x,y} I(x,y)e^{-j2\pi(ux+vy)/N}$ decomposes
it into those sinusoids. We usually view the **log-magnitude spectrum**
$\log(1+|F|)$ after `fftshift` (so the DC/low frequencies are at the centre and
high frequencies at the edges).

Run the cell below to build a synthetic image of crossing sinusoids.



In [ ]:
H, W = 256, 256
yy, xx = np.mgrid[0:H, 0:W]
# horizontal stripes of 8 cycles + vertical stripes of 4 cycles
img_sin = 0.5 * np.cos(2 * np.pi * 8 * yy / H) + 0.5 * np.cos(2 * np.pi * 4 * xx / W)
img_sin = (img_sin - img_sin.min()) / (img_sin.max() - img_sin.min())   # -> [0,1]

F = np.fft.fftshift(np.fft.fft2(img_sin))
mag = np.log1p(np.abs(F))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img_sin, cmap='gray'); ax[0].set_title("image (sum of 2 sinusoids)"); ax[0].axis('off')
ax[1].imshow(mag, cmap='inferno');  ax[1].set_title("log-magnitude spectrum (shifted)"); ax[1].axis('off')
plt.tight_layout(); plt.show()
print("Two bright pairs of dots = the two frequencies. The dots are symmetric → real image.")



### 🛠️ Task 1 — Where are the frequencies?
There are two symmetric bright-dot pairs: the pair on the vertical frequency axis corresponds to the 8-cycles-vertical sinusoid, while the pair on the horizontal axis corresponds to the horizontal sinusoid.

1. `fftshift` moves the zero-frequency (DC) component from a corner to the centre, making the spectrum easier to interpret visually.
2. The log compresses the spectrum's enormous dynamic range, so weaker frequency peaks remain visible alongside DC.


## Part 2 · Filtering in the frequency domain

**Recap.** By the **convolution theorem**, filtering in space equals *multiplying*
the spectra:
$$I * h \;\longleftrightarrow\; F(u,v)\cdot H(u,v)$$
A perfect (ideal) low-pass keeps all frequencies below a cut-off radius $r_0$ and
removes everything else — but the sharp cut-off creates **ringing** (Gibbs
phenomenon) in the spatial domain.



### 🛠️ Task 2 — Ideal low-pass & high-pass masks 
Implement the two mask builders (centre of the array is $(0,0)$ because we apply
them **after** `fftshift`). Then run the filtering demo.



In [ ]:
def ideal_lowpass(shape, cutoff):
    """Binary mask: 1 inside a disc of radius cutoff around the centre, else 0."""
    h, w = shape
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float64)
    return (((yy - h / 2)**2 + (xx - w / 2)**2) <= cutoff**2).astype(np.float64)


def ideal_highpass(shape, cutoff):
    """1 - ideal_lowpass."""
    return 1.0 - ideal_lowpass(shape, cutoff)


def apply_freq_filter(img, mask_centered):
    Fs = np.fft.fftshift(np.fft.fft2(img))
    return np.real(np.fft.ifft2(np.fft.ifftshift(Fs * mask_centered)))

mask = ideal_lowpass((H, W), 30)
low = apply_freq_filter(img_sin, mask)
print("low-pass mask sum (≈ π r² pixels):", int(mask.sum()))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for axx, r0 in zip(ax, [10, 30, 60]):
    axx.imshow(apply_freq_filter(img_sin, ideal_lowpass((H, W), r0)), cmap='gray')
    axx.set_title(f"ideal low-pass r0={r0}"); axx.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ✅ Self check — do not edit
m = ideal_lowpass((10, 10), 2)
assert m.shape == (10, 10) and set(np.unique(m)).issubset({0.0, 1.0}), "mask must be binary"
assert ideal_highpass((10, 10), 2).sum() == m.size - m.sum(), "highpass = 1 - lowpass"
print("✅ Task 2 assertions passed!")



### 📝 Task 3 — Reading the demo
1. As $r_0$ shrinks, high-frequency dot pairs are removed first, so their stripe pattern disappears; sharp cutoffs can add faint oscillatory ripples near transitions.
2. A brick-wall frequency cutoff has a sinc-like spatial impulse response with oscillatory side lobes (ringing); a Gaussian changes smoothly and has no sharp spectral discontinuity.


### 🛠️ Task 4 — Verify the convolution theorem 
Complete `fft_convolve2d` (pad the image with the kernel radius first so the FFT
multiplication does not wrap around), then check it equals spatial `cv2.filter2D`.



In [ ]:
def fft_convolve2d(img, kernel):
    """Convolve via FFT multiplication. img & kernel are 2-D float64."""
    kh, kw = kernel.shape; ph, pw = kh // 2, kw // 2
    h, w = img.shape
    padded = np.pad(img, ((ph, ph), (pw, pw)))
    big = np.zeros((h + 2*ph + kh - 1, w + 2*pw + kw - 1))
    big[:h + 2*ph, :w + 2*pw] = padded
    k = np.zeros_like(big); k[:kh, :kw] = kernel
    full = np.real(np.fft.ifft2(np.fft.fft2(big) * np.fft.fft2(k)))
    return full[kh-1:kh-1+h, kw-1:kw-1+w]

g = np.random.rand(40, 50)
kernel = np.ones((5, 5)) / 25.0
spatial = cv2.filter2D(g, cv2.CV_64F, kernel, borderType=cv2.BORDER_CONSTANT)
freq = fft_convolve2d(g, kernel)
print("max |spatial - fft| =", np.abs(spatial - freq).max())


In [ ]:
# ✅ Self check — do not edit
g = np.random.rand(40, 50)
kernel = np.ones((5, 5)) / 25.0
spatial = cv2.filter2D(g, cv2.CV_64F, kernel, borderType=cv2.BORDER_CONSTANT)
freq = fft_convolve2d(g, kernel)
assert np.allclose(spatial, freq, atol=1e-10), "convolution theorem is violated!"
print("✅ Task 4 assertions passed — space-domain and frequency-domain agree!")



## Part 3 · Image gradients & edge detection

**Recap.** The gradient $\nabla I=(I_x, I_y)$ is largest where the image changes
fastest (edges). Practical derivative filters such as **Sobel** combine smoothing
with differentiation:
$$G_x=\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix},\qquad
G_y=G_x^{\mathsf T}$$

Gradient **magnitude** $\|\nabla I\|=\sqrt{I_x^2+I_y^2}$ and **orientation**
$\theta=\operatorname{atan2}(I_y,I_x)$.



### 🛠️ Task 5 — Sobel magnitude & orientation 
Complete the function using the Sobel kernels above, then threshold the magnitude
into a binary edge map.



In [ ]:
def sobel_magnitude_orientation(img):
    """Return (magnitude, orientation_radians) of the image gradient (float64)."""
    sx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float64)
    sy = sx.T
    gx = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, sx)
    gy = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, sy)
    return np.hypot(gx, gy), np.arctan2(gy, gx)

H2, W2 = 200, 260
test = np.zeros((H2, W2), dtype=np.float64)
cv2.rectangle(test, (40,40), (140,150), 1.0, -1)
cv2.circle(test, (210,100), 40, .7, -1)
cv2.line(test, (10,180), (250,60), .9, 4)
mag, ori = sobel_magnitude_orientation(test)
gx = cv2.Sobel(test, cv2.CV_64F, 1, 0, ksize=3); gy = cv2.Sobel(test, cv2.CV_64F, 0, 1, ksize=3)
print("max |mine - cv2.Sobel| on magnitude =", np.abs(mag - np.hypot(gx, gy)).max())


In [ ]:
# ✅ Self check — do not edit
mag, ori = sobel_magnitude_orientation(test)
gx = cv2.Sobel(test, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(test, cv2.CV_64F, 0, 1, ksize=3)
mag_ref = np.hypot(gx, gy)
assert mag.shape == test.shape and ori.shape == test.shape
assert np.allclose(mag, mag_ref, atol=1e-6), "magnitude should match cv2.Sobel magnitude"
assert ori.min() >= -np.pi - 1e-9 and ori.max() <= np.pi + 1e-9, "orientation in [-pi, pi]"
print("✅ Task 5 assertions passed!")



### 🛠️ Task 6 — Edges from noisy images & DoG
Magnitude of raw derivatives amplifies **noise**. Fix: smooth first. Implement a
binary **edge map** by thresholding the magnitude of a *Gaussian-smoothed*
gradient, then compare qualitatively with `cv2.Canny`. Also build a **DoG** filter
(blur with $\sigma_1$ minus blur with $\sigma_2$) and show its band-pass response.



In [ ]:
def edge_map(mag, thresh):
    """Threshold a gradient magnitude into a 0/1 edge map."""
    return (np.asarray(mag) >= thresh).astype(np.uint8)

noisy = np.clip(test + np.random.normal(0, .08, test.shape), 0, 1)
blur = cv2.GaussianBlur(noisy, (5, 5), 1.2)
mag_smooth, _ = sobel_magnitude_orientation(blur)
edges = edge_map(mag_smooth, .12)
edges_canny = cv2.Canny(np.uint8(noisy * 255), 50, 150) > 0
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(noisy, cmap='gray'); ax[0].set_title("noisy image"); ax[0].axis('off')
ax[1].imshow(edges, cmap='gray'); ax[1].set_title("thresholded |∇(smooth I)|"); ax[1].axis('off')
ax[2].imshow(edges_canny, cmap='gray'); ax[2].set_title("cv2.Canny"); ax[2].axis('off')
plt.tight_layout(); plt.show()


### 📝 Task 7 — DoG reasoning
1. $\mathrm{DoG}_{\sigma_1,\sigma_2}=G_{\sigma_1}-G_{\sigma_2}$, with $\sigma_2>\sigma_1$; it is a band-pass filter that retains intermediate spatial frequencies.
2. Gaussian smoothing before differentiation suppresses noise; its $\sigma$ (and kernel size) controls the amount of suppression.


## Part 4 · Sampling, aliasing & pyramids

**Recap.** If we **subsample** an image without first removing the high frequencies,
those frequencies **alias** into lower ones and create fake patterns (Nyquist
theorem). A **Gaussian pyramid** iteratively low-passes and downsamples; the
**Laplacian pyramid** stores the detail lost at each level ($L_i=G_i-\mathrm{expand}(G_{i+1})$) and allows **perfect reconstruction**.



### 🛠️ Task 8 — Aliasing demo 
Subsample a high-frequency pattern by factor $k$ using plain slicing, and again
after a pre-smoothing. Observe the spurious low-frequency pattern (aliasing) in the
first case.



In [ ]:
# concentric rings = many high frequencies
yy, xx = np.mgrid[0:300, 0:300]
rings = np.cos(2 * np.pi * (np.hypot(xx - 150, yy - 150)) / 6.0)     # period 6 px
rings = (rings - rings.min()) / (rings.max() - rings.min())

k = 5
aliased = rings[::k, ::k]                                  # NO anti-alias filter
prefilt = cv2.GaussianBlur(rings, (0, 0), k / 2.0)         # low-pass first
nice = prefilt[::k, ::k]

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(rings, cmap='gray'); ax[0].set_title("original (period 6px)"); ax[0].axis('off')
ax[1].imshow(aliased, cmap='gray'); ax[1].set_title("subsample x%d (aliased!)" % k); ax[1].axis('off')
ax[2].imshow(nice, cmap='gray'); ax[2].set_title("smooth, then subsample"); ax[2].axis('off')
plt.tight_layout(); plt.show()



### 🛠️ Task 9 — Gaussian & Laplacian pyramids 
Complete `reduce` (blur then keep every other pixel) and `expand` (insert zeros,
then blur with the **doubled** kernel). Then run the provided `laplacian_pyramid`
and `reconstruct_from_laplacian` helpers (they call your reduce/expand) and check
perfect reconstruction.



In [ ]:
def gauss2d(a=0.4):
    """Burt-Adelson 5-tap kernel as a 2-D outer product."""
    w = np.array([.25-a/2, .25, a, .25, .25-a/2], dtype=np.float64)
    return np.outer(w, w)


def reduce(img):
    """Blur with gauss2d() then subsample by 2."""
    return cv2.filter2D(img.astype(np.float64), cv2.CV_64F, gauss2d(), borderType=cv2.BORDER_REFLECT)[::2, ::2]


def expand(img):
    """Upsample by 2 (insert zeros), then blur with 4 * gauss2d()."""
    h, w = img.shape
    up = np.zeros((2*h, 2*w)); up[::2, ::2] = img
    return cv2.filter2D(up, cv2.CV_64F, 4*gauss2d(), borderType=cv2.BORDER_REFLECT)


def laplacian_pyramid(img, levels):
    g = img.astype(np.float64); lap = []
    for _ in range(levels):
        g_next = reduce(g); lap.append(g - expand(g_next)); g = g_next
    return lap + [g]


def reconstruct_from_laplacian(lap):
    g = lap[-1]
    for L in reversed(lap[:-1]): g = expand(g) + L
    return g

H3, W3 = 160, 256
im = np.zeros((H3, W3), dtype=np.float64)
cv2.rectangle(im, (20,20), (90,130), .9, -1); cv2.circle(im, (190,80), 45, .5, -1); cv2.line(im, (30,140), (240,60), .3, 5)
gp = [im]
while gp[-1].shape[0] > 8 and gp[-1].shape[1] > 8: gp.append(reduce(gp[-1]))
lap = laplacian_pyramid(im, len(gp)-1); rec = reconstruct_from_laplacian(lap)
print("reconstruction max |error| =", np.abs(rec-im).max())


In [ ]:
# ✅ Self check — do not edit
H3, W3 = 160, 256
im = np.zeros((H3, W3), dtype=np.float64)
cv2.rectangle(im, (20, 20), (90, 130), 0.9, -1)
cv2.circle(im, (190, 80), 45, 0.5, -1)
cv2.line(im, (30, 140), (240, 60), 0.3, 5)
gp = [im]
while gp[-1].shape[0] > 8 and gp[-1].shape[1] > 8:
    gp.append(reduce(gp[-1]))
lap = laplacian_pyramid(im, len(gp) - 1)
rec = reconstruct_from_laplacian(lap)
assert np.allclose(rec, im, atol=1e-8), "Laplacian pyramid must reconstruct the image exactly"
print("✅ Task 9 assertions passed — perfect reconstruction!")



### 📝 Task 10 — Why pyramids?
1. Pyramids enable seamless image blending, compression, coarse-to-fine optical flow / registration, and multi-scale feature detection.
2. At a coarse level, a large image displacement becomes a small one, while examining multiple levels lets features be found at their appropriate scale.


## ✅ Lab 3 — Submission checklist

- [ ] Run All so every output is saved.
- [ ] Tasks 1 – 10 completed; all `Self check` cells print `✅ ... passed!`.
- [ ] Conceptual answers (Tasks 1, 3, 7, 10) written in the markdown boxes.
- [ ] Upload `Lab3_Frequency_Domain_Gradients_and_Image_Pyramids.ipynb`.

### 🔎 Reflection
Which concept (spectrum reading / frequency filtering / gradients / aliasing /
pyramids) would you like to revisit in the workshop?

> *Your reflection:*

